# 08 · Full 60-run backbone × regime × head × seed grid

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Run after seed-0 validation and the pilot-informed configuration freeze. This preserves the full resource-intensive design.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Write the run manifest

In [ ]:
from oncoplate.pipeline import grid_manifest,run_grid
manifest=grid_manifest(cfg);assert len(manifest)==60
write_table(p['reports']/"fit_registry.csv",manifest)
display(manifest)

## 2. Select a resumable queue for this runtime
Use disjoint run IDs across concurrent sessions. Never train the same run in two runtimes.

In [ ]:
RUN_IDS = ['resnet50_frozen_independent_s0','resnet50_frozen_joint_s0']
# After the smoke tests, replace with a named subset or:
# RUN_IDS = manifest.loc[manifest.status.ne('complete'),'run_id'].tolist()
print(manifest[manifest.run_id.isin(RUN_IDS)])

## 3. Execute the selected queue

In [ ]:
registry=run_grid(cfg,RUN_IDS)
write_table(p['reports']/"fit_registry.csv",registry)
display(registry[registry.run_id.isin(RUN_IDS)])

## 4. Review failures or resumable checkpoints without changing the test split

In [ ]:
from collections import Counter
print(Counter(registry.status))
print('Completed run folders retain best.pt, last.pt, run.json, validation predictions and per-epoch history.')
print('For interrupted runs, rerun the same queue with unchanged data/code/config. Never fabricate complete.json.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
